<div dir="rtl">

# اليوم الثاني — من البيانات الخام إلى أول نموذج
كل خلية تنفذ فكرة واحدة. شغّلها ثم اقرأ الناتج قبل الانتقال.

</div>

<div dir="rtl">

## المحطة 0 — تحميل بيانات منافذ

</div>

<div dir="rtl">

### 0.1 استيراد pandas

</div>

In [ ]:
import pandas as pd

<div dir="rtl">

### 0.2 تحديد رابط الملف

</div>

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'

<div dir="rtl">

### 0.3 قراءة الملف

</div>

In [ ]:
df = pd.read_csv(DATA_URL)
print('تم التحميل ✅')

<div dir="rtl">

# المحطة 1 — افهم الجدول

</div>

<div dir="rtl">

### 1.1 حجم الجدول

</div>

In [ ]:
print('عدد الصفوف والأعمدة:', df.shape)

<div dir="rtl">

### 1.2 أول خمس حالات

</div>

In [ ]:
display(df.head())

<div dir="rtl">

### 1.3 أسماء الأعمدة

</div>

In [ ]:
print(df.columns.tolist())

<div dir="rtl">

### 1.4 أنواع الأعمدة

</div>

In [ ]:
display(df.dtypes.to_frame(name="نوع البيانات"))

<div dir="rtl">

**ماذا تعلمنا؟** عرفنا حجم الجدول وأسماء الأعمدة وأنواعها قبل اتخاذ أي قرار تنظيف.

</div>

<div dir="rtl">

# المحطة 2 — افحص الجودة ونظّف نسخة من البيانات

</div>

<div dir="rtl">

### 2.1 حساب القيم المفقودة

</div>

In [ ]:
missing_count = df.isna().sum()
display(missing_count[missing_count > 0].to_frame('عدد المفقود'))

<div dir="rtl">

### 2.2 حساب نسبة القيم المفقودة

</div>

In [ ]:
missing_percent = df.isna().mean().mul(100).round(1)
display(missing_percent[missing_percent > 0].to_frame('نسبة المفقود %'))

<div dir="rtl">

### 2.3 إنشاء نسخة للتنظيف
نحافظ على `df` كما هو، ونجري التعديلات على `clean_df`.

</div>

In [ ]:
clean_df = df.copy()

<div dir="rtl">

### 2.4 إنشاء مؤشر لغياب التقييم
القيمة 1 تعني أن التقييم كان مفقودًا.

</div>

In [ ]:
clean_df['rating_missing'] = clean_df['avg_rating'].isna().astype(int)
display(clean_df[['avg_rating', 'rating_missing']].head())

<div dir="rtl">

### 2.5 تعويض التقييم بالوسيط

</div>

In [ ]:
rating_median = clean_df['avg_rating'].median()
clean_df['avg_rating'] = clean_df['avg_rating'].fillna(rating_median)
print('الوسيط المستخدم:', rating_median)

<div dir="rtl">

### 2.6 تعويض العرض المفقود بفئة واضحة

</div>

In [ ]:
clean_df['last_promo_used'] = clean_df['last_promo_used'].fillna('never_used')

<div dir="rtl">

### 2.7 فحص الصفوف المكررة

</div>

In [ ]:
duplicate_rows = clean_df.duplicated().sum()
print('عدد الصفوف المكررة:', duplicate_rows)

<div dir="rtl">

### 2.8 فحص نطاق التقييم

</div>

In [ ]:
invalid_rating = ~clean_df['avg_rating'].between(1, 5)
print('عدد التقييمات خارج 1 إلى 5:', invalid_rating.sum())

<div dir="rtl">

### 2.9 مشاهدة أعلى قيم السلة قبل اتخاذ القرار

</div>

In [ ]:
display(clean_df.nlargest(10, 'avg_basket_sar')[['customer_id', 'avg_basket_sar']])

<div dir="rtl">

**القرار المحلول:** نبقي القيمة المرتفعة؛ لأنها ممكنة وليست دليلًا كافيًا على خطأ إدخال.

</div>

<div dir="rtl">

# المحطة 3 — جهّز X وy ومنع التسريب

</div>

<div dir="rtl">

### 3.1 حفظ الهدف y أولًا

</div>

In [ ]:
y = clean_df['churned_30d'].copy()

<div dir="rtl">

### 3.2 تحديد الأعمدة التي لا تدخل في X

</div>

In [ ]:
drop_cols = [
    'customer_id', 'churned_30d',
    'refund_issued', 'support_ticket_after_snapshot', 'next_month_orders',
    'signup_date', 'snapshot_date'
]

<div dir="rtl">

### 3.3 إنشاء X

</div>

In [ ]:
X = clean_df.drop(columns=drop_cols, errors="ignore")

<div dir="rtl">

### 3.4 التأكد من الحجم

</div>

In [ ]:
print('X:', X.shape)
print('y:', y.shape)

<div dir="rtl">

# المحطة 4 — تقسيم البيانات وبناء Pipeline

</div>

<div dir="rtl">

### 4.1 تقسيم البيانات

</div>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

<div dir="rtl">

### 4.2 معرفة الأعمدة الرقمية

</div>

In [ ]:
numeric_cols = X.select_dtypes(include='number').columns.tolist()
print(numeric_cols)

<div dir="rtl">

### 4.3 معرفة الأعمدة الفئوية

</div>

In [ ]:
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()
print(categorical_cols)

<div dir="rtl">

### 4.4 مسار تجهيز الأرقام

</div>

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

<div dir="rtl">

### 4.5 مسار تجهيز الفئات

</div>

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])

<div dir="rtl">

### 4.6 جمع المسارين

</div>

In [ ]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer([
    ('numbers', numeric_pipe, numeric_cols),
    ('categories', categorical_pipe, categorical_cols)
])

<div dir="rtl">

# المحطة 5 — Baseline ثم نموذج تصنيف

</div>

<div dir="rtl">

### 5.1 إنشاء خط الأساس (Baseline)

</div>

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = Pipeline([
    ('preprocess', preprocess),
    ('model', DummyClassifier(strategy='most_frequent'))
])

<div dir="rtl">

### 5.2 تدريب خط الأساس

</div>

In [ ]:
baseline.fit(X_train, y_train)

<div dir="rtl">

### 5.3 قياس Accuracy لخط الأساس

</div>

In [ ]:
baseline_accuracy = baseline.score(X_test, y_test)
print('Baseline Accuracy:', round(baseline_accuracy, 3))

<div dir="rtl">

### 5.4 إنشاء Logistic Regression

</div>

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

<div dir="rtl">

### 5.5 تدريب النموذج

</div>

In [ ]:
logistic_model.fit(X_train, y_train)

<div dir="rtl">

### 5.6 إنتاج الفئات المتوقعة

</div>

In [ ]:
test_prediction = logistic_model.predict(X_test)

<div dir="rtl">

### 5.7 إنتاج احتمال الفئة 1

</div>

In [ ]:
test_probability = logistic_model.predict_proba(X_test)[:, 1]

<div dir="rtl">

### 5.8 مشاهدة أول عشر نتائج

</div>

In [ ]:
result = pd.DataFrame({
    'الحقيقة': y_test.iloc[:10].values,
    'التوقع': test_prediction[:10],
    'احتمال التوقف': test_probability[:10].round(3)
})
display(result)

<div dir="rtl">

## الخلاصة
مررنا بالترتيب: فهم الجدول ← فحص الجودة ← التنظيف ← تحديد X وy ← التقسيم ← تجهيز البيانات ← Baseline ← أول نموذج.

</div>